# COMAVI — Complex-Aware Variant Impact Scoring
### Plug-and-play Google Colab workflow for user-supplied missense variants

This notebook guides users from a gene and variant list to a validated COMAVI run. It supports two workflows:

1. **Create a new system:** retrieve or upload monomer structures, rank experimental complexes, infer chain assignments and residue-numbering offsets by sequence alignment, review a traffic-light preflight, and save the validated setup for reuse.
2. **Reuse prepared system bundle(s):** upload one or more previously validated COMAVI setup bundles and score new variants across those systems without rebuilding the structural configuration.

For each variant, COMAVI keeps two products distinct:

- **Priority score — integrated structural-disruption score (ISDS-v1):** ranks the strength of modeled structural-disruption evidence for follow-up.
- **Mechanism profile:** preserves signed isolated-protein, assembled-complex, and partner-binding hypotheses needed to choose an experiment.

The setup wizard automates sequence alignment, chain mapping, variant coverage, and uniform numbering-offset inference. It does **not** decide whether a structure represents the biologically relevant ligand, conformation, modification, oligomer, or disease state. The notebook therefore recommends a structure and explains its evidence, but asks the user to confirm biologically important choices.

> **Predict once, score many.** Save the generated `COMAVI_system_setup_bundle.zip`. Later, upload that bundle, change the variant list, and rerun the scoring steps without repeating structural setup.

Repository: https://github.com/la424/comavi

COMAVI predicts modeled structural disruption and a proposed mechanism. **Neither ISDS-v1 nor the mechanism profile is a pathogenicity verdict.**


## Before you start

COMAVI uses [FoldX](https://foldxsuite.crg.eu/) to calculate ΔΔG values. FoldX is **not bundled** with this notebook or repository.

1. Register and download FoldX under the licence appropriate for your use.
2. Obtain the **Linux x86-64** build because Google Colab runs Linux.
3. In the next cell, upload the binary or select a copy stored in Google Drive.

Without FoldX, the notebook can still parse variants, retrieve structures, rank chain mappings, infer numbering, and build reusable system bundles, but it cannot perform energetic calculations.

For new systems, an experimental biological assembly is preferred when it contains the relevant residues and state. Otherwise, upload an AlphaFold Server model. The optional ColabFold route depends on the current Colab GPU environment and may occasionally require upstream fixes.


In [ ]:
#@title Setup: clone the current public COMAVI release and install dependencies { display-mode: "form" }
import os, sys, subprocess, shutil, importlib, re
from pathlib import Path

# "main" gives public users the current verified release. Every result bundle
# records the exact resolved commit so the run remains reproducible. Users may
# replace "main" with a recorded 40-character commit for an exact rerun.
COMAVI_REF = "main"  #@param {type:"string"}

REPO = Path("/content/comavi")
PY = sys.executable

if not REPO.exists():
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/la424/comavi.git", str(REPO)],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO), "fetch", "-q", "--tags", "--prune", "origin"],
        check=True,
    )

ref = COMAVI_REF.strip() or "main"
if ref == "main":
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "--detach", ref], check=True)

resolved_commit = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

if re.fullmatch(r"[0-9a-fA-F]{40}", ref) and resolved_commit.lower() != ref.lower():
    raise RuntimeError(
        f"Requested COMAVI commit {ref}, but Git resolved {resolved_commit}."
    )

subprocess.run(
    [
        PY,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(REPO / "requirements.txt"),
        "gemmi",
        "matplotlib",
    ],
    check=True,
)

required_runtime_files = [
    REPO / "run.py",
    REPO / "notebooks" / "comavi_helpers.py",
    REPO / "notebooks" / "comavi_setup_wizard.py",
    REPO / "scripts" / "build_isds_variant_report.py",
    REPO / "verification" / "verify_isds_output_surfaces.py",
    REPO / "scripts" / "comavi_v7" / "isds.py",
]
missing_runtime_files = [str(path) for path in required_runtime_files if not path.is_file()]
if missing_runtime_files:
    raise FileNotFoundError(
        "The selected COMAVI ref lacks required public-workflow files:\n  "
        + "\n  ".join(missing_runtime_files)
    )

SCRIPTS = REPO / "scripts"
for directory in (str(REPO / "notebooks"), str(SCRIPTS)):
    if directory not in sys.path:
        sys.path.insert(0, directory)

import comavi_helpers
importlib.reload(comavi_helpers)
H = comavi_helpers

import comavi_setup_wizard
importlib.reload(comavi_setup_wizard)
W = comavi_setup_wizard

from comavi_v7.isds import ISDS_OUTPUT_COLUMNS, ISDS_VERSION
ISDS_FIELDS = list(ISDS_OUTPUT_COLUMNS)
EXPECTED_ISDS_FIELD_SET = {
    "isds_version",
    "isds_available",
    "isds_v1",
    "isds_energy_ratio_uncapped",
    "isds_energy_component",
    "isds_context_component",
    "isds_dominant_axis",
    "isds_dominant_partner",
    "isds_dominant_signed_ddg",
}
if set(ISDS_FIELDS) != EXPECTED_ISDS_FIELD_SET:
    raise RuntimeError(
        "The selected COMAVI ref exposes an unexpected ISDS output contract: "
        + repr(ISDS_FIELDS)
    )

WORK = Path("/content/work")
STRUCT = WORK / "structures"
STRUCT.mkdir(parents=True, exist_ok=True)
OUT = WORK / "out"
OUT.mkdir(parents=True, exist_ok=True)

gpu = subprocess.run(
    ["bash", "-lc", "nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null || echo none"],
    capture_output=True,
    text=True,
).stdout.strip()

print("COMAVI repository:", REPO)
print("Resolved commit:", resolved_commit)
print("Setup wizard:", W.SETUP_WIZARD_VERSION)
print("ISDS version:", ISDS_VERSION)
print("ISDS fields:", len(ISDS_FIELDS))
print("GPU:", gpu if gpu and gpu != "none" else "none (needed only for optional ColabFold)")
print("Setup OK.")


In [ ]:
#@title Supply your FoldX Linux binary { display-mode: "form" }
FOLDX_SOURCE = "Upload now"  #@param ["Upload now", "Google Drive path"]
FOLDX_DRIVE_PATH = "/content/drive/MyDrive/foldx"  #@param {type:"string"}

from google.colab import files

if FOLDX_SOURCE == "Google Drive path":
    from google.colab import drive
    drive.mount("/content/drive")
    source_path = Path(FOLDX_DRIVE_PATH).expanduser()
    if not source_path.is_file():
        raise FileNotFoundError(
            f"No FoldX file exists at {source_path}. Update FOLDX_DRIVE_PATH and rerun."
        )
else:
    print("Select your LINUX x86-64 FoldX binary.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No FoldX binary was uploaded.")
    source_name = next(iter(uploaded))
    source_path = Path(source_name)

FOLDX = WORK / "foldx"
if FOLDX.exists():
    FOLDX.unlink()
shutil.copy2(source_path, FOLDX)
os.chmod(FOLDX, 0o755)
os.environ["FOLDX_BINARY"] = str(FOLDX)

magic = FOLDX.read_bytes()[:4]
ELF = bytes([0x7f, 0x45, 0x4c, 0x46])
MACHO = (
    bytes([0xcf, 0xfa, 0xed, 0xfe]),
    bytes([0xce, 0xfa, 0xed, 0xfe]),
    bytes([0xca, 0xfe, 0xba, 0xbe]),
    bytes([0xfe, 0xed, 0xfa, 0xcf]),
)
if magic == ELF:
    check = subprocess.run([str(FOLDX), "-h"], capture_output=True, text=True, timeout=60)
    if check.returncode not in (0, 1):
        raise RuntimeError("Linux FoldX binary was detected but did not run correctly.")
    print("OK: Linux binary detected and runnable.")
    print("FOLDX_BINARY =", FOLDX)
elif magic in MACHO:
    raise RuntimeError(
        "That is a macOS FoldX binary. Download the Linux x86-64 build and rerun."
    )
elif magic[:2] == bytes([0x4d, 0x5a]):
    raise RuntimeError(
        "That is a Windows FoldX binary. Download the Linux x86-64 build and rerun."
    )
else:
    raise RuntimeError(
        "The selected file does not look like a Linux ELF executable."
    )


---
## Step 1 — Choose a setup mode and define variants

- **Create a new system** guides you through monomer retrieval, experimental-complex ranking, predicted-complex upload, automatic chain mapping, numbering inference, and a traffic-light preflight.
- **Use prepared system bundle(s)** accepts one or more `COMAVI_system_setup_bundle.zip` files. This is the easiest route for collaborators and supports a mixed variant table spanning multiple previously prepared systems. The stored structures and chain identities are reused, but every new variant is re-aligned and the combined numbering offsets are rebuilt for the current batch before FoldX.

Variants can be entered inline as `"GENE AAchange"` or uploaded as a CSV with at least `gene`, `ref_aa`, `position`, and `alt_aa`. Extra columns are preserved.

`INPUT_NUMBERING_OVERRIDES` is an optional fallback for variant conventions that differ from canonical reference numbering, written as `GENE:OFFSET`. The wizard first tries to infer this value from the submitted reference amino acids and reports ambiguity instead of silently choosing a residue.


In [ ]:
#@title Project, variants, and protein system { display-mode: "both" }
SETUP_MODE = "Create a new system"  #@param ["Create a new system", "Use prepared system bundle(s)"]
USE_DEMO = True  #@param {type:"boolean"}
VARIANT_INPUT_MODE = "Inline list"  #@param ["Inline list", "Upload CSV"]
HUB_GENE = "SHROOM3"  #@param {type:"string"}
PARTNERS = "ROCK2"  #@param {type:"string"}
MONOMER_SOURCE = "AlphaFold DB (auto-fetch)"  #@param ["AlphaFold DB (auto-fetch)", "Manual upload"]
ORGANISM_ID = 9606  #@param {type:"integer"}
INPUT_NUMBERING_OVERRIDES = ""  #@param {type:"string"}

VARIANTS = [
    "SHROOM3 G1003R",
    "SHROOM3 T1012N",
]

import pandas as pd
from IPython.display import display, HTML
from google.colab import files

prepared_mode = SETUP_MODE == "Use prepared system bundle(s)"
prepared_merge_manifest = None
setup_bundle_paths = []

if prepared_mode:
    USE_DEMO = False
    print("Upload one or more COMAVI_system_setup_bundle.zip files.")
    uploaded_bundles = files.upload()
    setup_bundle_paths = [
        Path(name)
        for name in uploaded_bundles
        if name.lower().endswith(".zip")
    ]
    if not setup_bundle_paths:
        raise RuntimeError("No setup bundle ZIP was uploaded.")
    prepared_root = WORK / "prepared_systems"
    cfg_path, STRUCT, prepared_merge_manifest = W.merge_setup_bundles(
        setup_bundle_paths,
        prepared_root,
    )
    prepared_reference_sequences = W.read_fasta(
        prepared_root / prepared_merge_manifest["reference_sequences"]
    )
    OUT = WORK / "out"
    OUT.mkdir(parents=True, exist_ok=True)
    genes_all = W.genes_from_config(cfg_path)
    configured_genes = set(genes_all)
    print("Prepared systems:", prepared_merge_manifest["systems"])
    print("Configured genes:", genes_all)
else:
    if USE_DEMO:
        HUB_GENE = "TNNI3"
        PARTNERS = "TNNC1"
        MONOMER_SOURCE = "AlphaFold DB (auto-fetch)"
        VARIANT_INPUT_MODE = "Inline list"
        VARIANTS = ["TNNI3 R162W", "TNNI3 P82S"]
        print(
            "DEMO MODE: TNNI3 + TNNC1 with two distinct substitutions. "
            "Uncheck USE_DEMO for your own genes."
        )
    partner_list = [
        value.strip()
        for value in PARTNERS.replace(",", " ").split()
        if value.strip()
    ]
    genes_all = [HUB_GENE] + partner_list
    configured_genes = {gene.lower() for gene in genes_all}

if VARIANT_INPUT_MODE == "Upload CSV" and not USE_DEMO:
    print("Upload a CSV with gene, ref_aa, position, and alt_aa columns.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No variant CSV was uploaded.")
    uploaded_name = next(iter(uploaded))
    vdf = pd.read_csv(uploaded_name, encoding="utf-8-sig")
    vdf.columns = [str(column).strip() for column in vdf.columns]
    verr = []
else:
    vdf, verr = H.parse_variants("\n".join(VARIANTS))

required_variant_columns = ["gene", "ref_aa", "position", "alt_aa"]
missing_variant_columns = [
    column for column in required_variant_columns if column not in vdf.columns
]
if missing_variant_columns:
    raise ValueError(
        "Variant input is missing required columns: "
        + ", ".join(missing_variant_columns)
    )

vdf = vdf.copy()
vdf["gene"] = vdf["gene"].astype(str).str.strip().str.lower()
vdf["ref_aa"] = vdf["ref_aa"].astype(str).str.strip().str.upper()
vdf["alt_aa"] = vdf["alt_aa"].astype(str).str.strip().str.upper()
vdf["position"] = pd.to_numeric(vdf["position"], errors="raise").astype(int)
if "variant" not in vdf.columns:
    vdf["variant"] = vdf["ref_aa"] + vdf["position"].astype(str) + vdf["alt_aa"]
else:
    vdf["variant"] = vdf["variant"].astype(str).str.strip()

if vdf.empty:
    raise ValueError("No valid variants were parsed.")
invalid_aa = vdf.loc[
    ~vdf["ref_aa"].str.fullmatch(r"[ACDEFGHIKLMNPQRSTVWY]")
    | ~vdf["alt_aa"].str.fullmatch(r"[ACDEFGHIKLMNPQRSTVWY]")
]
if not invalid_aa.empty:
    raise ValueError(
        "Only standard one-letter amino-acid codes are accepted. Invalid rows:\n"
        + invalid_aa[["gene", "variant"]].to_string(index=False)
    )
unknown_genes = sorted(set(vdf["gene"]) - configured_genes)
if unknown_genes:
    raise ValueError(
        "These variant genes are absent from the selected system bundle(s) or hub/partner definition: "
        + ", ".join(unknown_genes)
    )

input_offset_overrides = W.parse_gene_integer_map(
    INPUT_NUMBERING_OVERRIDES,
    configured_genes=configured_genes,
    label="INPUT_NUMBERING_OVERRIDES",
)

print(f"Parsed {len(vdf)} variants across {vdf['gene'].nunique()} gene(s).")
display(vdf)
if verr:
    print("Skipped inline entries:")
    for line_number, raw, reason in verr:
        print("  line", line_number, repr(raw), "->", reason)
print("VARIANT INPUT CONTRACT: PASS")


---
## Step 2a — Retrieve or upload monomer structures

The wizard aligns each monomer chain to the corresponding reference sequence, checks submitted reference amino acids, and derives the numbering offset used by COMAVI. A red result stops the workflow. A yellow result explains what needs confirmation.

This step is skipped when prepared system bundles are used.


In [ ]:
#@title Retrieve and map monomer structures { display-mode: "form" }
import shutil

if prepared_mode:
    gene_uniprot, gene_seq, gene_monomer, monomer_assessments = {}, {}, {}, {}
    print("Prepared-system mode: monomer structures and numbering are already stored in the bundle(s).")
else:
    gene_uniprot, gene_seq, gene_monomer = {}, {}, {}
    try:
        ORGANISM_ID = int(str(ORGANISM_ID).strip())
    except (ValueError, TypeError):
        ORGANISM_ID = 9606
        print("ORGANISM_ID was invalid; defaulting to 9606 (human).")

    if MONOMER_SOURCE == "AlphaFold DB (auto-fetch)":
        for gene in genes_all:
            try:
                accession, candidates = H.resolve_uniprot(gene, organism_id=ORGANISM_ID)
            except Exception as error:
                print("Could not resolve", gene, "on UniProt:", error)
                continue
            if not accession:
                print("WARNING: no reviewed UniProt entry for", gene, "— use Manual upload.")
                continue
            try:
                path = H.fetch_alphafold_monomer(accession, STRUCT)
                gene_key = gene.lower()
                gene_uniprot[gene_key] = accession
                gene_monomer[gene_key] = path
                gene_seq[gene_key] = H.fetch_uniprot_sequence(accession)
                print(f"{gene:10s} {accession}  {H.count_residues(path)} residues -> {path.name}")
            except Exception as error:
                print("Resolved", gene, "to", accession, "but could not fetch its model:", error)
    else:
        print("Upload one monomer PDB/CIF per gene; start each filename with the gene symbol.")
        uploaded = files.upload()
        for source_name in list(uploaded):
            destination = STRUCT / source_name
            shutil.move(source_name, destination)
            stem = destination.stem.lower()
            match = next((gene.lower() for gene in genes_all if stem.startswith(gene.lower())), None)
            if match:
                gene_monomer[match] = destination
                print(f"{match:10s} <- {destination.name} ({H.count_residues(destination)} residues)")
            else:
                print("Could not map", destination.name, "to a gene; rename it to begin with the gene symbol.")
        for gene in genes_all:
            gene_key = gene.lower()
            try:
                accession, _ = H.resolve_uniprot(gene, organism_id=ORGANISM_ID)
                if accession:
                    gene_uniprot[gene_key] = accession
                    gene_seq[gene_key] = H.fetch_uniprot_sequence(accession)
            except Exception:
                pass

    missing_sequences = [gene for gene in configured_genes if gene not in gene_seq]
    missing_monomers = [gene for gene in configured_genes if gene not in gene_monomer]
    if missing_sequences:
        raise RuntimeError("Reference sequences are missing for: " + ", ".join(missing_sequences))
    if missing_monomers:
        raise RuntimeError("Monomer structures are missing for: " + ", ".join(missing_monomers))

    monomer_assessments = W.assess_monomer_set(
        gene_monomer,
        gene_seq,
        vdf,
        input_offset_overrides=input_offset_overrides,
    )
    monomer_table = W.monomer_assessment_frame(monomer_assessments)
    print("Automatic monomer sequence and numbering assessment:")
    display(monomer_table)
    if (monomer_table["status"] == "red").any():
        raise RuntimeError(
            "MONOMER PREFLIGHT: STOP. Review the red rows, reference sequence, isoform, and structure file."
        )
    print("MONOMER PREFLIGHT: PASS WITH REVIEW" if (monomer_table["status"] == "yellow").any() else "MONOMER PREFLIGHT: PASS")


---
## Step 2b-0 — Rank experimental complexes

For a new system, the notebook searches RCSB, downloads a small candidate set, aligns every configured gene to every observed chain, checks coverage of the submitted variants, and ranks candidate chain assignments.

A **green** candidate has strong sequence mapping and covers every submitted variant. A **yellow** candidate needs review. A **red** candidate is not safe to score. Resolution and sequence mapping help rank structures, but they cannot establish that a deposited assembly contains the biologically relevant ligand, conformation, modification, or disease state. Confirm that scientific context before accepting a recommendation.


In [ ]:
#@title Search, rank, and optionally select an experimental complex { display-mode: "form" }
MAX_HITS = 40  #@param {type:"integer"}
ASSESS_TOP_HITS = 5  #@param {type:"integer"}
EXPERIMENTAL_PDB_ID = ""  #@param {type:"string"}
EXPERIMENTAL_CHAIN_MAP = ""  #@param {type:"string"}
ACCEPT_RECOMMENDED_EXPERIMENTAL = False  #@param {type:"boolean"}

experimental_complex_pdb = None
experimental_gene_chain = None
experimental_assessment = None
candidate_table = pd.DataFrame()
candidate_assessments = {}

if prepared_mode:
    print("Prepared-system mode: experimental-complex selection is already stored in the bundle(s).")
else:
    hits = H.rcsb_find_and_rank(list(genes_all), max_hits=MAX_HITS, inspect=max(12, ASSESS_TOP_HITS))
    candidates = []
    candidate_directory = WORK / "candidate_pdbs"
    candidate_directory.mkdir(parents=True, exist_ok=True)
    for record in hits[: max(0, int(ASSESS_TOP_HITS))]:
        try:
            candidate_path = Path(H.fetch_pdb_entry(record["pdb_id"], candidate_directory))
            candidate = dict(record)
            candidate["path"] = candidate_path
            candidates.append(candidate)
        except Exception as error:
            print("Could not download candidate", record.get("pdb_id"), ":", error)

    if candidates:
        candidate_table, candidate_assessments = W.rank_structure_candidates(
            candidates,
            gene_seq,
            vdf,
            input_offset_overrides=input_offset_overrides,
        )
        display_columns = [
            column
            for column in (
                "recommended_rank", "candidate_id", "status", "score", "chain_map",
                "pipeline_offsets", "resolution_A", "method", "year", "title", "message"
            )
            if column in candidate_table.columns
        ]
        print("Sequence-aware experimental-structure ranking:")
        display(candidate_table[display_columns])
    else:
        print("No downloadable experimental candidate was available. Continue with a predicted complex.")

    recommended_id = None
    if not candidate_table.empty:
        safe = candidate_table.loc[candidate_table["status"].isin(["green", "yellow"])]
        if not safe.empty:
            recommended_id = str(safe.iloc[0]["candidate_id"])
            print("Recommended candidate for review:", recommended_id)

    selected_id = EXPERIMENTAL_PDB_ID.strip().upper()
    if not selected_id and ACCEPT_RECOMMENDED_EXPERIMENTAL:
        selected_id = recommended_id or ""
        if selected_id:
            print("Using the recommended candidate after explicit acceptance:", selected_id)

    if selected_id:
        if selected_id in candidate_assessments:
            experimental_assessment = candidate_assessments[selected_id]
            selected_row = candidate_table.loc[candidate_table["candidate_id"].eq(selected_id)].iloc[0]
            candidate_source = Path(selected_row["path"])
            experimental_complex_pdb = STRUCT / candidate_source.name
            shutil.copy2(candidate_source, experimental_complex_pdb)
        else:
            experimental_complex_pdb = Path(H.fetch_pdb_entry(selected_id, STRUCT))
            chain_overrides = W.parse_gene_text_map(
                EXPERIMENTAL_CHAIN_MAP,
                configured_genes=configured_genes,
                label="EXPERIMENTAL_CHAIN_MAP",
            )
            experimental_assessment = W.assess_structure(
                experimental_complex_pdb,
                gene_seq,
                vdf,
                source_label=selected_id,
                input_offset_overrides=input_offset_overrides,
                chain_overrides=chain_overrides,
            )
        if EXPERIMENTAL_CHAIN_MAP.strip():
            chain_overrides = W.parse_gene_text_map(
                EXPERIMENTAL_CHAIN_MAP,
                configured_genes=configured_genes,
                label="EXPERIMENTAL_CHAIN_MAP",
            )
            experimental_assessment = W.assess_structure(
                experimental_complex_pdb,
                gene_seq,
                vdf,
                source_label=selected_id,
                input_offset_overrides=input_offset_overrides,
                chain_overrides=chain_overrides,
            )
        experimental_gene_chain = experimental_assessment.chain_map
        print("Selected experimental complex:", experimental_complex_pdb)
        print("Automatic gene-to-chain map:", experimental_gene_chain)
        display(W.assessment_assignment_frame(experimental_assessment))
        if experimental_assessment.status == "red":
            raise RuntimeError(
                "The selected experimental complex failed sequence/coverage mapping. Choose another candidate."
            )
    else:
        print("No experimental PDB selected. Continue to the predicted-complex routes.")


---
## Step 2b — Provide or predict the complex and run the setup preflight

Choose an experimental PDB selected above, upload an AlphaFold Server result, or use optional ColabFold. After a structure is available, the wizard automatically:

- assigns genes to chains by sequence alignment;
- infers input, monomer, and multimer numbering offsets;
- checks every submitted reference amino acid in both structural contexts;
- detects non-uniform numbering that cannot be represented by one COMAVI offset;
- builds a traffic-light preflight and actionable correction messages.

Manual chain and offset fields remain available as auditable overrides. They are fallbacks, not the default workflow. Before a new system can run, the user must also confirm that the selected coordinates represent the biologically relevant partners, construct, state, ligand/cofactor, modification, and oligomer.


In [ ]:
#@title Provide/predict the complex and map it automatically { display-mode: "form" }
MULTIMER_MODE = "Auto (recommended)"  #@param ["Auto (recommended)", "Experimental PDB selected above", "AlphaFold Server (upload .cif)", "ColabFold (GPU)", "Domain-scoped ColabFold"]
HUB_DOMAIN_RANGE = ""  #@param {type:"string"}
COLABFOLD_RELEASE = "v1.6.1"  #@param {type:"string"}
COMPLEX_CHAIN_OVERRIDES = ""  #@param {type:"string"}
MONOMER_OFFSET_OVERRIDES = ""  #@param {type:"string"}
MULTIMER_OFFSET_OVERRIDES = ""  #@param {type:"string"}
CONFIRM_YELLOW_MAPPING = False  #@param {type:"boolean"}

if prepared_mode:
    complex_pdb = None
    structure_type = None
    gene_chain = {}
    mono_off = {}
    multi_off = {}
    complex_assessment = None
    print("Prepared-system mode: complex structure, chain map, and offsets are loaded from bundle config(s).")
else:
    chain_overrides = W.parse_gene_text_map(
        COMPLEX_CHAIN_OVERRIDES,
        configured_genes=configured_genes,
        label="COMPLEX_CHAIN_OVERRIDES",
    )
    monomer_pipeline_overrides = W.parse_gene_integer_map(
        MONOMER_OFFSET_OVERRIDES,
        configured_genes=configured_genes,
        label="MONOMER_OFFSET_OVERRIDES",
    )
    multimer_pipeline_overrides = W.parse_gene_integer_map(
        MULTIMER_OFFSET_OVERRIDES,
        configured_genes=configured_genes,
        label="MULTIMER_OFFSET_OVERRIDES",
    )

    mode = MULTIMER_MODE
    if mode == "Auto (recommended)":
        if experimental_complex_pdb is not None and experimental_assessment is not None:
            mode = "Experimental PDB selected above"
            print("Auto -> selected experimental PDB.")
        else:
            mode = "AlphaFold Server (upload .cif)"
            print("Auto -> AlphaFold Server upload route.")

    hub = HUB_GENE.lower()
    domain_multimer_offsets = {}
    sys_name = "_".join(gene.lower() for gene in genes_all)
    complex_pdb = None
    structure_type = "AF"

    if mode == "Experimental PDB selected above":
        if experimental_complex_pdb is None or experimental_assessment is None:
            raise RuntimeError("No experimental complex is ready. Return to Step 2b-0.")
        complex_pdb = Path(experimental_complex_pdb)
        structure_type = "PDB"
    else:
        sequences = []
        for gene in genes_all:
            sequence = gene_seq.get(gene.lower(), "")
            if not sequence:
                raise RuntimeError(f"No reference sequence is available for {gene}.")
            if gene.lower() == hub and HUB_DOMAIN_RANGE.strip():
                start, end = [int(value) for value in HUB_DOMAIN_RANGE.replace("-", " ").split()]
                sequence = H.slice_sequence(sequence, start, end)
                domain_multimer_offsets[hub] = start - 1
                print(
                    f"Domain-scoping {HUB_GENE} to {start}-{end} ({len(sequence)} aa)."
                )
            sequences.append(sequence)
        total_residues = sum(len(sequence) for sequence in sequences)
        print(f"Total predicted-complex size: approximately {total_residues} residues.")

        if mode == "AlphaFold Server (upload .cif)":
            print("Open https://alphafoldserver.com and create one protein entity per gene in this order:")
            for index, (gene, sequence) in enumerate(zip(genes_all, sequences), start=1):
                print(f"  entity {index}: {gene} ({len(sequence)} aa)")
            sequence_file = WORK / f"{sys_name}_sequences.txt"
            sequence_file.write_text(
                "\n".join(f">{gene}\n{sequence}" for gene, sequence in zip(genes_all, sequences)),
                encoding="utf-8",
            )
            print("Upload the downloaded .cif or .zip result.")
            uploaded = files.upload()
            if not uploaded:
                raise RuntimeError("No AlphaFold Server result was uploaded.")
            source_name = next(iter(uploaded))
            raw_path = WORK / source_name
            shutil.move(source_name, raw_path)
            if raw_path.suffix.lower() == ".zip":
                import zipfile
                with zipfile.ZipFile(raw_path) as archive:
                    cif_names = [name for name in archive.namelist() if name.lower().endswith(".cif")]
                    if not cif_names:
                        raise RuntimeError("The uploaded ZIP contains no CIF file.")
                    archive.extract(cif_names[0], WORK)
                    raw_path = WORK / cif_names[0]
            complex_pdb = STRUCT / f"{sys_name}.pdb"
            H.cif_to_pdb(raw_path, complex_pdb)
        elif mode in ("ColabFold (GPU)", "Domain-scoped ColabFold"):
            gpu_name = subprocess.run(
                ["bash", "-lc", "nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null"],
                capture_output=True,
                text=True,
            ).stdout.strip()
            if not gpu_name:
                raise RuntimeError(
                    "No GPU is attached. Select Runtime > Change runtime type > GPU or use AlphaFold Server."
                )
            import glob as _glob
            release_token = COLABFOLD_RELEASE.replace("/", "_").replace(".", "_")
            ready = Path(f"/content/COLABFOLD_READY_{release_token}")
            if not ready.exists():
                package = (
                    "colabfold[alphafold-minus-jax] @ "
                    f"git+https://github.com/sokrypton/ColabFold@{COLABFOLD_RELEASE}"
                )
                install = subprocess.run(
                    [PY, "-m", "pip", "install", "-q", "--no-warn-conflicts", package],
                    capture_output=True,
                    text=True,
                )
                if install.returncode != 0:
                    raise RuntimeError("ColabFold installation failed. Use AlphaFold Server.\n" + install.stderr[-3500:])
                download = subprocess.run([PY, "-m", "colabfold.download"], capture_output=True, text=True)
                if download.returncode != 0:
                    raise RuntimeError("ColabFold parameter download failed.\n" + download.stderr[-3500:])
                ready.touch()
            fasta = H.write_fasta(sys_name, sequences, WORK / "cf_in")
            cf_out = WORK / "cf_out"
            cf_out.mkdir(exist_ok=True)
            run = subprocess.run(
                ["colabfold_batch", "--num-models", "1", str(fasta), str(cf_out)],
                capture_output=True,
                text=True,
            )
            if run.returncode != 0:
                raise RuntimeError("ColabFold failed. Use AlphaFold Server.\n" + run.stderr[-3500:])
            models = (
                _glob.glob(str(cf_out / "*relaxed_rank_001*.pdb"))
                or _glob.glob(str(cf_out / "*rank_001*.pdb"))
                or _glob.glob(str(cf_out / "*.pdb"))
            )
            if not models:
                raise RuntimeError("ColabFold completed but produced no PDB file.")
            complex_pdb = STRUCT / f"{sys_name}.pdb"
            shutil.copy(sorted(models)[0], complex_pdb)
        else:
            raise ValueError(f"Unsupported MULTIMER_MODE: {mode}")

    if complex_pdb is None or not Path(complex_pdb).is_file():
        raise RuntimeError("No usable complex structure was produced or selected.")

    complex_assessment = W.assess_structure(
        complex_pdb,
        gene_seq,
        vdf,
        source_label=mode,
        input_offset_overrides=input_offset_overrides,
        chain_overrides=chain_overrides,
        pipeline_offset_overrides=multimer_pipeline_overrides,
    )
    gene_chain = complex_assessment.chain_map
    multi_off = complex_assessment.pipeline_offsets.copy()
    mono_off = {
        gene: assessment.pipeline_offset
        for gene, assessment in monomer_assessments.items()
        if assessment.pipeline_offset is not None
    }
    mono_off.update(monomer_pipeline_overrides)

    print("Automatic complex chain and numbering assessment:")
    display(W.assessment_assignment_frame(complex_assessment))
    display(W.assessment_variant_frame(complex_assessment))
    print("Final gene-to-chain map:", gene_chain)
    print("Final monomer offsets:", mono_off)
    print("Final multimer offsets:", multi_off)

    if complex_assessment.status == "red":
        raise RuntimeError("COMPLEX PREFLIGHT: STOP. Resolve the red mapping rows before FoldX.")
    if complex_assessment.status == "yellow" and not CONFIRM_YELLOW_MAPPING:
        raise RuntimeError(
            "COMPLEX PREFLIGHT: REVIEW. Read the yellow messages, then check CONFIRM_YELLOW_MAPPING and rerun if the mapping is scientifically appropriate."
        )
    print("AUTOMATIC CHAIN AND NUMBERING MAPPING: PASS")


In [ ]:
#@title Build the configuration, traffic-light preflight, and reusable setup bundle { display-mode: "form" }
CONFIRM_YELLOW_PREFLIGHT = False  #@param {type:"boolean"}
CONFIRM_BIOLOGICAL_CONTEXT = False  #@param {type:"boolean"}
DOWNLOAD_SETUP_BUNDLE = True  #@param {type:"boolean"}

if prepared_mode:
    cfg_path, preflight, prepared_system_reports = W.revalidate_prepared_systems(
        cfg_path,
        STRUCT,
        prepared_reference_sequences,
        vdf,
        output_config_path=WORK / "prepared_systems" / "config_for_current_variants.yaml",
        input_offset_overrides=input_offset_overrides,
    )
    print("Prepared systems were revalidated against the current variant batch.")
    print("Current-batch configuration:", cfg_path)
    try:
        H.validate_config(cfg_path, SCRIPTS)
    except Exception as error:
        raise RuntimeError(f"Prepared COMAVI configuration failed validation: {error}") from error
    display(HTML(W.render_traffic_light_html(preflight, title="Prepared-system current-variant preflight")))
    display(preflight)
    if (preflight["overall"] == "red").any():
        raise RuntimeError(
            "PREPARED-SYSTEM PREFLIGHT: STOP. At least one current variant does not map safely to the stored structures."
        )
    if (preflight["overall"] == "yellow").any() and not CONFIRM_YELLOW_PREFLIGHT:
        raise RuntimeError(
            "PREPARED-SYSTEM PREFLIGHT: REVIEW. Read the yellow explanations, then check CONFIRM_YELLOW_PREFLIGHT and rerun if appropriate."
        )
    setup_bundle_path = None
else:
    missing_monomers = [gene for gene in genes_all if gene.lower() not in gene_monomer]
    missing_chains = [gene for gene in genes_all if gene.lower() not in gene_chain]
    if missing_monomers:
        raise RuntimeError("Missing monomer structures for: " + ", ".join(missing_monomers))
    if missing_chains:
        raise RuntimeError("Missing complex-chain assignments for: " + ", ".join(missing_chains))
    if not CONFIRM_BIOLOGICAL_CONTEXT:
        raise RuntimeError(
            "BIOLOGICAL CONTEXT CONFIRMATION REQUIRED. Confirm that the selected complex contains the relevant partners, construct, ligand/cofactor state, conformation, modification, and oligomer for your question; then check CONFIRM_BIOLOGICAL_CONTEXT and rerun."
        )

    genes_block = []
    for gene in genes_all:
        gene_key = gene.lower()
        genes_block.append(
            {
                "gene": gene_key,
                "chain": gene_chain[gene_key],
                "monomer_file": str(gene_monomer[gene_key]),
                "monomer_offset": int(mono_off.get(gene_key, 0)),
                "multimer_offset": int(multi_off.get(gene_key, 0)),
            }
        )
    systems = [
        {
            "name": sys_name,
            "structure_type": structure_type,
            "complex_file": str(complex_pdb),
            "genes": genes_block,
        }
    ]
    cfg_path = H.build_config_yaml(systems, WORK / "config.yaml")
    print(cfg_path.read_text(encoding="utf-8"))
    try:
        H.validate_config(cfg_path, SCRIPTS)
    except Exception as error:
        raise RuntimeError(f"COMAVI configuration validation failed: {error}") from error

    preflight = W.build_preflight_table(vdf, monomer_assessments, complex_assessment)
    display(HTML(W.render_traffic_light_html(preflight)))
    display(preflight)
    if (preflight["overall"] == "red").any():
        raise RuntimeError("SYSTEM PREFLIGHT: STOP. Resolve every red row before FoldX.")
    if (preflight["overall"] == "yellow").any() and not CONFIRM_YELLOW_PREFLIGHT:
        raise RuntimeError(
            "SYSTEM PREFLIGHT: REVIEW. Read the yellow explanations, then check CONFIRM_YELLOW_PREFLIGHT and rerun if appropriate."
        )

    setup_report = {
        "setup_wizard_version": W.SETUP_WIZARD_VERSION,
        "comavi_commit_at_setup": resolved_commit,
        "system": sys_name,
        "status": "yellow" if (preflight["overall"] == "yellow").any() else "green",
        "complex_assessment": W.assessment_to_jsonable(complex_assessment),
        "monomer_assessments": {
            gene: W.assessment_to_jsonable(value)
            for gene, value in monomer_assessments.items()
        },
        "biological_context_confirmed_by_user": True,
        "biological_context_confirmation_required_for_interpretation": True,
        "note": (
            "Sequence mapping cannot establish that the selected structure contains the biologically relevant ligand, state, modification, or oligomer."
        ),
    }
    setup_bundle_path = W.create_system_setup_bundle(
        WORK / "COMAVI_system_setup_bundle.zip",
        config_path=cfg_path,
        structure_paths=[*gene_monomer.values(), Path(complex_pdb)],
        reference_sequences=gene_seq,
        preflight=preflight,
        setup_report=setup_report,
    )
    print("Reusable setup bundle:", setup_bundle_path)
    if DOWNLOAD_SETUP_BUNDLE:
        files.download(str(setup_bundle_path))

print("CONFIGURATION CONTRACT: PASS")
print("TRAFFIC-LIGHT PREFLIGHT: PASS")


---
## Step 3 — Run COMAVI

This step invokes the generic `run.py` entry point. It supports a single newly configured system or multiple prepared system bundles. The runner repeats residue-identity and structure-provenance checks before FoldX, so the setup wizard does not replace the engine's safety checks.

After scoring, the notebook presents plain-language result cards, the complete mechanism profile, the priority table, plots, machine-readable outputs, logs, provenance, and the reusable setup bundle.


In [ ]:
#@title Run COMAVI structural assessment { display-mode: "form" }
_need = ("vdf", "FOLDX", "WORK", "cfg_path", "STRUCT")
_missing = [name for name in _need if name not in globals()]
assert not _missing, "Undefined: " + ", ".join(_missing) + " — rerun the preceding cells."

if not Path(FOLDX).is_file():
    raise FileNotFoundError("The FoldX binary is missing. Rerun the FoldX cell.")
variant_csv = WORK / "variants.csv"
vdf.to_csv(variant_csv, index=False)
# Colab cells may be rerun. Clear generated results so FoldX run records and
# uncertainty statistics cannot accumulate across invocations.
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)

environment = dict(os.environ)
environment["FOLDX_BINARY"] = str(FOLDX)
command = [
    PY,
    str(REPO / "run.py"),
    "--config", str(cfg_path),
    "--variants", str(variant_csv),
    "--structures", str(STRUCT),
    "--out", str(OUT),
    "--foldx", str(FOLDX),
]
print("Running:", " ".join(command))
process = subprocess.run(command, env=environment, capture_output=True, text=True)
(OUT / "run_stdout.log").write_text(process.stdout, encoding="utf-8")
(OUT / "run_stderr.log").write_text(process.stderr, encoding="utf-8")
if process.stdout.strip():
    print(process.stdout[-5000:])
if process.returncode != 0:
    print("STDERR tail:\n", process.stderr[-5000:])
    raise RuntimeError(f"COMAVI exited with status {process.returncode}.")
res_csv = OUT / "structural_results.csv"
if not res_csv.is_file():
    raise FileNotFoundError(f"COMAVI completed without producing {res_csv}.")
print("Raw results:", res_csv)
print("COMAVI RUN: PASS")


In [ ]:
#@title Verify, review, and download the priority score and mechanism profile { display-mode: "form" }
from IPython.display import display, HTML
import json, zipfile
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

_need = ("res_csv", "vdf", "ISDS_FIELDS")
_missing = [name for name in _need if name not in globals()]
assert not _missing, "Undefined: " + ", ".join(_missing) + " — rerun the COMAVI cell."

sdf = pd.read_csv(res_csv, low_memory=False)
if sdf.empty:
    raise RuntimeError("The COMAVI result table is empty.")

required_raw_columns = ["gene", "ref_aa", "position", "alt_aa", "ddg_monomer", "comavi_tier"]
missing_raw_columns = [column for column in required_raw_columns if column not in sdf.columns]
missing_isds_columns = [column for column in ISDS_FIELDS if column not in sdf.columns]
fold_columns = [column for column in sdf.columns if column.startswith("ddg_fold_") and not any(token in column for token in ("_sd", "_runs", "_ci95_", "_vote_"))]
binding_columns = [column for column in sdf.columns if column.startswith("ddg_binding_") and not any(token in column for token in ("_sd", "_runs", "_ci95_", "_vote_"))]
mechanism_columns = [column for column in sdf.columns if column.startswith("comavi_mechanism")]

contract_failures = []
if missing_raw_columns:
    contract_failures.append("missing raw fields: " + ", ".join(missing_raw_columns))
if missing_isds_columns:
    contract_failures.append("missing ISDS-v1 fields: " + ", ".join(missing_isds_columns))
if not fold_columns:
    contract_failures.append("no assembled-complex fold column")
if not binding_columns:
    contract_failures.append("no partner-binding column")
if not mechanism_columns:
    contract_failures.append("no mechanism-call column")

input_variant_ids = set(
    zip(
        vdf["gene"].astype(str).str.lower(),
        vdf["ref_aa"].astype(str).str.upper(),
        vdf["position"].astype(int),
        vdf["alt_aa"].astype(str).str.upper(),
    )
)
output_variant_ids = set(
    zip(
        sdf["gene"].astype(str).str.lower(),
        sdf["ref_aa"].astype(str).str.upper(),
        pd.to_numeric(sdf["position"], errors="raise").astype(int),
        sdf["alt_aa"].astype(str).str.upper(),
    )
)
missing_variants = sorted(input_variant_ids - output_variant_ids)
if missing_variants:
    contract_failures.append(f"submitted variants absent from output: {missing_variants}")

if contract_failures:
    raise RuntimeError("ARBITRARY-VARIANT OUTPUT CONTRACT: FAIL\n  " + "\n  ".join(contract_failures))

print(f"Rows: {len(sdf)} | submitted variants represented: {len(input_variant_ids)}/{len(input_variant_ids)}")
print(f"Assembled-complex columns: {len(fold_columns)} | binding columns: {len(binding_columns)}")
print("ISDS version values:", sorted(set(sdf["isds_version"].dropna().astype(str))))
print("ARBITRARY-VARIANT OUTPUT CONTRACT: PASS")

plain_language = W.plain_language_summary_frame(sdf)
print("Plain-language review cards:")
display(HTML(W.render_plain_language_cards(sdf)))
plain_language_csv = OUT / "comavi_plain_language_summary.csv"
plain_language.to_csv(plain_language_csv, index=False)

summary = H.per_partner_table(sdf)
print("Detailed COMAVI mechanism cards:")
display(HTML(H.render_cards_html(sdf)))
print("Per-variant × partner mechanism summary:")
display(summary)

available = sdf["isds_available"].fillna(False).astype(str).str.lower().isin({"true", "1", "1.0", "yes"})
priority_columns = [
    column
    for column in (
        "gene",
        "variant",
        "system",
        "isds_available",
        "isds_v1",
        "isds_energy_component",
        "isds_context_component",
        "isds_dominant_axis",
        "isds_dominant_partner",
        "isds_dominant_signed_ddg",
        "comavi_tier",
        "comavi_mechanism_t10",
        "comavi_mechanism_t25",
        "comavi_mechanism",
    )
    if column in sdf.columns
]
priority_table = sdf.loc[available, priority_columns].copy()
if "isds_v1" in priority_table.columns:
    priority_table = priority_table.sort_values("isds_v1", ascending=False, kind="mergesort")

print("Priority score table (ISDS-v1 is a prioritization index, not a probability):")
if priority_table.empty:
    print("No row had sufficient evaluable energy and tier information for ISDS-v1.")
else:
    display(priority_table)

plot_frame = summary.copy()
plot_frame["label"] = plot_frame["gene"].str.upper() + " " + plot_frame["variant"]
axis_plot = plot_frame.groupby("label").agg(
    monomer=("ddg_monomer", "first"),
    assembled_complex=("ddg_fold", lambda values: np.nanmax(np.abs(values)) if values.notna().any() else np.nan),
    binding=("ddg_binding", lambda values: np.nanmax(np.abs(values)) if values.notna().any() else np.nan),
)
axis = axis_plot.plot(kind="barh", figsize=(7, 0.5 * len(axis_plot) + 1))
axis.set_xlabel("maximum |ΔΔG| (kcal/mol)")
axis.set_title("Per-variant structural effect by modeled axis")
plt.tight_layout()
axis_png = OUT / "per_variant_axes.png"
plt.savefig(axis_png, dpi=140)
plt.show()

mechanism_summary_csv = OUT / "comavi_mechanism_summary.csv"
summary.to_csv(mechanism_summary_csv, index=False)

report_dir = OUT / "public_report"
report_dir.mkdir(parents=True, exist_ok=True)
report_prefix = "comavi_variants"
report_command = [
    PY,
    str(REPO / "scripts" / "build_isds_variant_report.py"),
    str(res_csv),
    "--out-dir",
    str(report_dir),
    "--prefix",
    report_prefix,
    "--top-n",
    "50",
]
report_process = subprocess.run(report_command, capture_output=True, text=True)
if report_process.stdout.strip():
    print(report_process.stdout[-2500:])
if report_process.returncode != 0:
    raise RuntimeError("Public report generation failed:\n" + report_process.stderr[-3500:])

full_report_csv = report_dir / f"{report_prefix}_with_isds_v1.csv"
prioritized_csv = report_dir / f"{report_prefix}_prioritized.csv"
summary_json = report_dir / f"{report_prefix}_isds_summary.json"
markdown_report = report_dir / f"{report_prefix}_isds_report.md"

verify_command = [
    PY,
    str(REPO / "verification" / "verify_isds_output_surfaces.py"),
    str(full_report_csv),
]
verify_process = subprocess.run(verify_command, capture_output=True, text=True)
print(verify_process.stdout.strip())
if verify_process.returncode != 0:
    raise RuntimeError("ISDS output verification failed:\n" + verify_process.stderr[-3500:])

report_summary = json.loads(summary_json.read_text(encoding="utf-8"))
print("Report summary:")
print(json.dumps(report_summary, indent=2))

config_provenance = W.build_config_provenance(cfg_path)

def _compact_provenance_text(value):
    if isinstance(value, str):
        return value
    return json.dumps(value, sort_keys=True, separators=(",", ":"))

release_record = OUT / "COMAVI_run_provenance.txt"
release_record.write_text(
    "\n".join(
        [
            f"comavi_commit={resolved_commit}",
            f"isds_version={ISDS_VERSION}",
            f"setup_wizard_version={W.SETUP_WIZARD_VERSION}",
            f"setup_mode={SETUP_MODE}",
            f"input_variants={len(vdf)}",
            f"output_rows={len(sdf)}",
            f"isds_available_rows={int(available.sum())}",
            "monomer_offsets=" + _compact_provenance_text(config_provenance["monomer_offsets"]),
            "multimer_offsets=" + _compact_provenance_text(config_provenance["multimer_offsets"]),
            "complex_structure_type=" + _compact_provenance_text(config_provenance["complex_structure_type"]),
            "complex_structure=" + _compact_provenance_text(config_provenance["complex_structure"]),
            "gene_chain_map=" + _compact_provenance_text(config_provenance["gene_chain_map"]),
            "system_configurations=" + _compact_provenance_text(config_provenance["system_configurations"]),
        ]
    )
    + "\n",
    encoding="utf-8",
)

bundle_path = WORK / "COMAVI_variant_results_bundle.zip"
bundle_files = [
    res_csv,
    mechanism_summary_csv,
    plain_language_csv,
    full_report_csv,
    prioritized_csv,
    summary_json,
    markdown_report,
    axis_png,
    cfg_path,
    WORK / "variants.csv",
    OUT / "run_stdout.log",
    OUT / "run_stderr.log",
    release_record,
]
if globals().get("setup_bundle_path") is not None and Path(setup_bundle_path).is_file():
    bundle_files.append(Path(setup_bundle_path))
if globals().get("preflight") is not None:
    preflight_csv = OUT / "comavi_setup_preflight.csv"
    preflight.to_csv(preflight_csv, index=False)
    bundle_files.append(preflight_csv)
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in bundle_files:
        if Path(path).is_file():
            archive.write(path, arcname=Path(path).name)

from google.colab import files
print("Downloading verified result bundle:", bundle_path)
files.download(str(bundle_path))


---
## Step 4 (optional) — Four-way concordance

Overlay your **manually-curated** AlphaMissense + Franklin/ClinVar calls next to the structural result. This is the only step that needs hand-entered data; skip it if you only want the structural assessment.

In [ ]:
#@title (Optional) Build the concordance template  { display-mode: "form" }
ctpl = H.concordance_template(vdf)
ctpl_path = OUT / "concordance_template.csv"
ctpl.to_csv(ctpl_path, index=False)
print("Fill the three columns (AM pathogenicity, AM class, franklin) per variant.")
print("Option A: edit ANNOTATIONS inline in the next cell. Option B: edit this CSV and upload it.")
display(ctpl)
from google.colab import files as _f
_f.download(str(ctpl_path))

In [ ]:
#@title (Optional) Run four-way concordance { display-mode: "form" }
ANNOTATION_MODE = "Upload filled template"  #@param ["Upload filled template", "Use ANNOTATIONS dict below"]

ANNOTATIONS = [
    # {"gene":"shroom3", "variant":"G1003R", "AM pathogenicity":0.81,
    #  "AM class":"likely_pathogenic", "franklin":"Likely Pathogenic"},
]

if ANNOTATION_MODE == "Upload filled template":
    from google.colab import files
    print("Upload a filled CSV/XLSX with gene, variant, AM pathogenicity, AM class, and franklin columns.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No annotation file was uploaded.")
    source_name = next(iter(uploaded))
    annotations = (
        pd.read_excel(source_name)
        if source_name.lower().endswith((".xlsx", ".xls"))
        else pd.read_csv(source_name)
    )
else:
    annotations = pd.DataFrame(ANNOTATIONS)

if annotations.empty:
    print("No annotations provided — skipping concordance.")
else:
    master, concordance_process = H.run_concordance(
        res_csv,
        annotations,
        OUT / "concordance",
        SCRIPTS,
        python_exe=PY,
    )
    print(concordance_process.stdout[-1500:])
    if concordance_process.returncode != 0:
        raise RuntimeError("Concordance failed:\n" + concordance_process.stderr[-2500:])
    if master and master.exists():
        concordance = pd.read_csv(master, low_memory=False)
        keep = [
            column
            for column in (
                "gene",
                "variant",
                "isds_available",
                "isds_v1",
                "isds_dominant_axis",
                "comavi_tier",
                "comavi_mechanism_t10",
                "comavi_mechanism_t25",
                "comavi_mechanism",
                "AM pathogenicity",
                "AM class",
                "franklin",
            )
            if column in concordance.columns
        ]
        display(concordance[keep])
        from google.colab import files as download_files
        download_files.download(str(master))
    else:
        raise RuntimeError("Concordance completed without an output file.")

---
## Notes, scope, and troubleshooting

- **Fastest collaborator workflow:** prepare and review a system once, save `COMAVI_system_setup_bundle.zip`, and share that bundle. Collaborators can upload it with a new variant CSV and skip structure selection. The notebook still rechecks each new variant and rebuilds numbering offsets for the current input convention.
- **Multi-system batches:** upload multiple prepared setup bundles. The notebook merges their YAML configurations and structures, then lets the generic runner fan variants into matching systems.
- **Safe automation:** sequence alignment, chain assignment, variant coverage, and uniform offsets are automated. Red mappings stop. Yellow mappings require explicit confirmation.
- **Biological context is not automatable from coordinates alone.** Confirm that the structure represents the relevant partner, ligand, conformation, modification, oligomer, and construct before interpreting a silent result.
- **Uniform-offset limit:** the current COMAVI engine represents numbering with one integer offset per gene and structural context. A construct with internal insertions, deletions, or non-uniform renumbering may require a preprocessed structure and should remain red until resolved.
- **Homomers and repeated copies:** the current setup wizard assigns one configured gene to one chain. Symmetric homomers or multiple copies of the same gene require expert review and are not yet a one-click path.
- **FoldX is required and not bundled.** Upload a licensed Linux build or select it from Google Drive.
- **GPU is optional.** It is needed only for ColabFold. AlphaFold Server upload and experimental-PDB routes do not require a GPU.
- **Two outputs, two uses:** use ISDS-v1 to allocate structural follow-up; use the signed mechanism profile to choose stability, assembly, or binding experiments.
- **ISDS-v1 is not a probability.** A silent or unavailable result does not establish benignity.
- **Exact reproducibility:** every result bundle records the resolved COMAVI commit and setup-wizard version. Replace `COMAVI_REF="main"` with that commit to rerun the same code later.

Repository: https://github.com/la424/comavi
